## Task 1: Build a Fault-Tolerant Multi-Source ETL Pipeline with Conflict Resolution

### 1. Extract data from JSONPlaceholder API

In [66]:
import requests
import pandas as pd
import time

def fetch_data_with_retries(url, retries=3, delay=5):
    for i in range(retries):
        try:
            response = requests.get(url, timeout=10) # 10 second timeout
            response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
            return response.json()
        except requests.exceptions.Timeout:
            print(f"Request timed out for {url}. Retrying ({i+1}/{retries})...")
        except requests.exceptions.ConnectionError as e:
            print(f"Connection error for {url}: {e}. Retrying ({i+1}/{retries})...")
        except requests.exceptions.HTTPError as e:
            print(f"HTTP error for {url}: {e}. Retrying ({i+1}/{retries})...")
        except Exception as e:
            print(f"An unexpected error occurred for {url}: {e}. Retrying ({i+1}/{retries})...")
        time.sleep(delay)
    print(f"Failed to fetch data from {url} after {retries} retries.")
    return None

users_url = 'https://jsonplaceholder.typicode.com/users'
users_data = fetch_data_with_retries(users_url)

if users_data:
    users_df = pd.json_normalize(users_data)
    print("Users data extracted and normalized successfully:")
    display(users_df.head())
else:
    print("Could not retrieve users data.")

Users data extracted and normalized successfully:


,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems


In [67]:
posts_url = 'https://jsonplaceholder.typicode.com/posts'
posts_data = fetch_data_with_retries(posts_url)

if posts_data:
    posts_df = pd.DataFrame(posts_data)
    print("Posts data extracted successfully:")
    display(posts_df.head())
else:
    print("Could not retrieve posts data.")

Posts data extracted successfully:


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


### 2. Extract data from a locally generated messy CSV file

In [68]:
import io

# Create a dummy messy CSV file content
csv_data = """
id,Name,Email,Age,City,  Country,  Salary
1,  John Doe,john.doe@example.com,30,New York,USA,60000
2,Jane Smith  ,jane.smith@example.com, 25,  Los Angeles, USA ,75000
3,Peter Jones,peter.jones@example.com,40,London  ,UK,80000
4,  Alice Brown,alice.brown@example.com, 35,Paris  ,France,  62000
5,Bob White  ,bob.white@example.com,  50,  Berlin,Germany,  90000
6,  Charlie Green,charlie.green@example.com,,Sydney  ,Australia,
7,Diana Prince,diana.prince@example.com,28,Themyscira,USA,70000
"""

# Read the CSV data into a DataFrame
try:
    csv_df = pd.read_csv(io.StringIO(csv_data))
    print("CSV data extracted successfully:")
    display(csv_df.head())
except Exception as e:
    print(f"Error reading CSV data: {e}")

CSV data extracted successfully:


,id,Name,Email,Age,City,Country,Salary
0,1,John Doe,john.doe@example.com,30.0,New York,USA,60000.0
1,2,Jane Smith,jane.smith@example.com,25.0,Los Angeles,USA,75000.0
2,3,Peter Jones,peter.jones@example.com,40.0,London,UK,80000.0
3,4,Alice Brown,alice.brown@example.com,35.0,Paris,France,62000.0
4,5,Bob White,bob.white@example.com,50.0,Berlin,Germany,90000.0


### 3. Normalize and Merge DataFrames with Conflict Resolution

In [69]:
# Clean column names in csv_df (remove leading/trailing spaces, convert to lowercase)
csv_df.columns = csv_df.columns.str.strip().str.lower()

# Rename 'email' column in csv_df for consistency if necessary (already lowercase from above)
# csv_df = csv_df.rename(columns={'email': 'email'}) # This is now redundant due to .str.lower()

# Display cleaned csv_df columns
print("Cleaned CSV DataFrame columns:")
print(csv_df.columns)

# Merge the two DataFrames on 'email'
# We'll perform a full outer merge to keep all records from both DFs initially
merged_df = pd.merge(users_df, csv_df, on='email', how='outer', suffixes=('_api', '_csv'))

# Conflict Resolution for 'name': If 'email' matches, prioritize 'name' from API data.
# Create a new 'name' column based on priority
merged_df['name'] = merged_df['name_api'].fillna(merged_df['name_csv'])

# Drop the original conflicting 'name' columns
merged_df = merged_df.drop(columns=['name_api', 'name_csv'])

print("\nMerged DataFrame with conflict resolution for 'name' (API data prioritized):")
display(merged_df.head())

# Display rows where 'email' was present in both, to show conflict resolution
conflicted_emails = merged_df[merged_df['email'].isin(users_df['email']) & merged_df['email'].isin(csv_df['email'])]
if not conflicted_emails.empty:
    print("\nRows with potentially conflicted 'name' values (API name prioritized):")
    display(conflicted_emails.head())
else:
    print("\nNo direct email conflicts with different names found in the merged data.")

Cleaned CSV DataFrame columns:
Index(['id', 'name', 'email', 'age', 'city', 'country', 'salary'], dtype='object')

Merged DataFrame with conflict resolution for 'name' (API data prioritized):


,id_api,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs,id_csv,age,city,country,salary,name
0,9.0,Delphine,Chaim_McDermott@dana.io,(775)976-6794 x41206,conrad.com,Dayna Park,Suite 449,Bartholomebury,76495-3109,24.6463,-168.8889,Yost and Sons,Switchable contextually-based project,aggregate real-time technologies,NaN,NaN,NaN,NaN,NaN,Glenna Reichert
1,4.0,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services,NaN,NaN,NaN,NaN,NaN,Patricia Lebsack
2,6.0,Leopoldo_Corkery,Karley_Dach@jasper.info,1-477-935-8478 x6430,ola.org,Norberto Crossing,Apt. 950,South Christy,23505-1337,-71.4197,71.7478,Considine-Lockman,Synchronised bottom-line interface,e-enable innovative applications,NaN,NaN,NaN,NaN,NaN,Mrs. Dennis Schulist
3,5.0,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems,NaN,NaN,NaN,NaN,NaN,Chelsey Dietrich
4,3.0,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications,NaN,NaN,NaN,NaN,NaN,Clementine Bauch



No direct email conflicts with different names found in the merged data.


### 5. Load the final unified dataset to CSV and MySQL

In [70]:
# Load to CSV
output_csv_path = 'cleaned_unified_data.csv'
merged_df.to_csv(output_csv_path, index=False)
print(f"Cleaned data saved to {output_csv_path}")

Cleaned data saved to cleaned_unified_data.csv


In [71]:

from sqlalchemy import create_engine
import os
import dotenv
import pymysql # Required for MySQL connector
from urllib.parse import quote_plus
from sqlalchemy import text


dotenv.load_dotenv()

# MySQL connection details (replace with your credentials)
mysql_user = 'root'
mysql_password = quote_plus(os.getenv("password"))
mysql_host = 'localhost'
mysql_port = 3306
mysql_database = 'etl_pipeline_db'

# Create a SQLAlchemy engine
try:
    # Use pymysql as the DBAPI for mysqlclient

    
    engine = create_engine(f'mysql+pymysql://{mysql_user}:{mysql_password}@{mysql_host}:{mysql_port}/{mysql_database}')

    # Check if connection is successful by trying to connect
    with engine.connect() as connection:
        print("Successfully connected to MySQL database.")

    # Load data to MySQL
    # On a second run, no duplicate rows should be inserted.
    # We'll use a temporary table and then perform an INSERT IGNORE or ON DUPLICATE KEY UPDATE
    # First, ensure the 'email' column is suitable for a unique index in MySQL
    merged_df['email'] = merged_df['email'].astype(str).str.lower().str.strip()

    # Create a unique index on 'email' in the target MySQL table to handle duplicates.
    # If the table doesn't exist, create it with 'email' as a unique key.
    # If it exists, try to add the unique key if it's not already there.

    # Option 1: Using to_sql with if_exists='append' and relying on a unique index in MySQL
    # This requires the table and unique index to be pre-created or handled carefully.
    # For this example, let's create the table schema and then insert.

    table_name = 'unified_data'

    # Convert columns to types suitable for MySQL
    # Pandas default integer type for nullable integers is Int64, which is object in Python, convert to float then int.
    # We need to explicitly handle float64 to int for columns like age, salary, id_api, id_csv
    df_to_sql = merged_df.copy()
    for col in ['id_api', 'id_csv', 'age', 'salary']:
        if col in df_to_sql.columns:
            df_to_sql[col] = pd.to_numeric(df_to_sql[col], errors='coerce')
            df_to_sql[col] = df_to_sql[col].fillna(0).astype(int) # Fill NaNs with 0 before converting to int

    # Create table if not exists with unique constraint on email
    create_table_sql = f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        id_api BIGINT,
        username VARCHAR(255),
        email VARCHAR(255) PRIMARY KEY,
        phone VARCHAR(255),
        website VARCHAR(255),
        address_street VARCHAR(255),
        address_suite VARCHAR(255),
        address_city VARCHAR(255),
        address_zipcode VARCHAR(255),
        address_geo_lat VARCHAR(255),
        address_geo_lng VARCHAR(255),
        company_name VARCHAR(255),
        company_catchPhrase TEXT,
        company_bs TEXT,
        id_csv BIGINT,
        age INT,
        city VARCHAR(255),
        country VARCHAR(255),
        salary INT,
        name VARCHAR(255)
    );
    """

    with engine.connect() as connection:
        connection.execute(text(create_table_sql))
        connection.commit() # Commit the DDL operation
    print(f"Table '{table_name}' ensured to exist with 'email' as PRIMARY KEY.")

    # Use 'if_exists="append"' and 'method=None' to allow SQLAlchemy to use default INSERT behavior,
    # and MySQL's PRIMARY KEY constraint on 'email' will prevent duplicates.
    # Rows with existing emails will cause an error unless handled with ON DUPLICATE KEY UPDATE or INSERT IGNORE.
    # Pandas to_sql does not directly support INSERT IGNORE or ON DUPLICATE KEY UPDATE in all cases.
    # A common workaround is to use a temporary table and then merge.

    # Simplified approach for demonstration: directly insert and rely on PRIMARY KEY error.
    # For true idempotency with updates on conflict, a custom function for to_sql or direct SQL is needed.

    # Renaming columns for MySQL compatibility (remove dots)
    df_to_sql.columns = df_to_sql.columns.str.replace('.', '_')

    # Insert data, ignoring duplicates (this is a more robust way to handle 'no duplicate rows should be inserted')
    # We need to build the SQL INSERT IGNORE statement manually as pandas to_sql doesn't support it directly.
    # Alternatively, use 'to_sql' with a temporary table and then a SQL MERGE/UPSERT.

    # Let's use a custom method for to_sql to achieve INSERT IGNORE functionality
    def to_sql_insert_ignore(df, table_name, con, if_exists, index, chunksize=None):
        from sqlalchemy.dialects import mysql
        insert_statements = []
        for _, row in df.iterrows():
            # Convert row to dict, handling NaT/NaN for non-string types for SQL
            row_dict = row.to_dict()
            for k, v in row_dict.items():
                if pd.isna(v):
                    row_dict[k] = None

            # Create an insert statement
            ins = mysql.insert(con.table(table_name, autoload_with=con)).values(row_dict)
            # Compile to an INSERT IGNORE statement
            on_duplicate_key_stmt = ins.on_duplicate_key_update(
                # You can specify which columns to update if there's a conflict
                # For this task, "no duplicate rows should be inserted" implies just ignoring
                # Or if you want to update some fields:
                # name=ins.inserted.name, age=ins.inserted.age # etc.
            )
            # We want to ignore, so we just use the plain insert, and rely on PRIMARY KEY to prevent duplicates,
            # which will raise an IntegrityError. If we want to *silently ignore* duplicates, we need INSERT IGNORE.
            # Unfortunately, sqlalchemy's on_duplicate_key_update is for UPSERT, not INSERT IGNORE.
            # For strict 'INSERT IGNORE', direct SQL is often cleaner.

            # Fallback to direct SQL for INSERT IGNORE
            columns = ', '.join([f'`{col}`' for col in df.columns])
            values_placeholders = ', '.join(['%s'] * len(df.columns))
            value_list = [row_dict[col] for col in df.columns]
            insert_statements.append(f"INSERT IGNORE INTO `{table_name}` ({columns}) VALUES ({values_placeholders});")

        with con.begin() as connection:
            for i, stmt in enumerate(insert_statements):
                connection.execute(stmt, df.iloc[i].tolist()) # Use row's values directly
        print(f"Successfully inserted/ignored {len(df)} rows into '{table_name}'.")

    # Correcting column names for SQLAlchemy: remove special characters or dots
    df_to_sql.columns = df_to_sql.columns.str.replace('.', '_', regex=False) # Use regex=False for literal dot
    df_to_sql = df_to_sql.where(pd.notnull(df_to_sql), None)

    # Define the data types for SQLAlchemy to create the table correctly if it doesn't exist
    from sqlalchemy.types import String, BigInteger, Integer, Text
    dtype_mapping = {
        'id_api': BigInteger,
        'username': String(255),
        'email': String(255),
        'phone': String(255),
        'website': String(255),
        'address_street': String(255),
        'address_suite': String(255),
        'address_city': String(255),
        'address_zipcode': String(255),
        'address_geo_lat': String(255), # Store as string for flexibility given mixed types
        'address_geo_lng': String(255), # Store as string for flexibility given mixed types
        'company_name': String(255),
        'company_catchPhrase': Text,
        'company_bs': Text,
        'id_csv': BigInteger,
        'age': Integer,
        'city': String(255),
        'country': String(255),
        'salary': Integer,
        'name': String(255)
    }

    # Filter dtype_mapping to only include columns present in df_to_sql
    filtered_dtype_mapping = {col: dtype_mapping[col] for col in df_to_sql.columns if col in dtype_mapping}

    # Use to_sql with if_exists='append' and a custom method to handle INSERT IGNORE
    # Since the custom method needs to be implemented separately or use raw SQL, we'll use raw SQL for INSERT IGNORE

    # Function to insert data using INSERT IGNORE
    def insert_ignore_into_mysql(df, table_name, engine):
        connection = engine.raw_connection()
        cursor = connection.cursor()
        df = df.where(pd.notnull(df), None)

        columns = ', '.join([f'`{col}`' for col in df.columns])
        placeholders = ', '.join(['%s'] * len(df.columns))
        insert_sql = f"INSERT IGNORE INTO `{table_name}` ({columns}) VALUES ({placeholders})"

        data = [tuple(row) for row in df.values]

        try:
            cursor.executemany(insert_sql, data)
            connection.commit()
            print(f"Successfully inserted/ignored {len(data)} rows into '{table_name}'.")
        except Exception as e:
            connection.rollback()
            print(f"Error inserting data: {e}")
        finally:
            cursor.close()
            connection.close()

    insert_ignore_into_mysql(df_to_sql, table_name, engine)

except Exception as e:
    print(f"Error connecting to or loading data into MySQL: {e}")
    print("Please ensure MySQL is running and credentials are correct.")

Successfully connected to MySQL database.
Table 'unified_data' ensured to exist with 'email' as PRIMARY KEY.
Successfully inserted/ignored 17 rows into 'unified_data'.


### 4. Apply 6 Cleaning Techniques

In [72]:
import numpy as np # Import numpy for np.nan

print("\n--- Before Cleaning ---")
print(merged_df.info())
print("\nMissing values before cleaning:")
print(merged_df.isnull().sum()[merged_df.isnull().sum() > 0])

# 1. Handle Nulls:
# For numerical columns (age, salary), fill with median/mean. For simplicity, using median here.
# For categorical/text columns, fill with 'Unknown' or mode.

# Identify numerical and categorical columns for null handling
numerical_cols = ['age', 'salary']
categorical_cols = ['city', 'country', 'username', 'phone', 'website', 'address.street', 'address.suite', 'address.city', 'address.zipcode', 'company.name', 'company.catchPhrase', 'company.bs', 'name']

for col in numerical_cols:
    if col in merged_df.columns:
        # First, ensure all values in object columns that should be numeric are clean
        if merged_df[col].dtype == 'object':
            # Replace whitespace-only strings with NaN before converting to numeric
            merged_df[col] = merged_df[col].replace(r'^\s*$', np.nan, regex=True)

        # Convert to numeric, coercing errors, then fill NaNs with the median
        # The median should be calculated *after* potential non-numeric values are coerced to NaN
        merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')
        median_val = merged_df[col].median()
        merged_df[col] = merged_df[col].fillna(median_val)

for col in categorical_cols:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].fillna('Unknown')


# 2. Handle Duplicates:
# Drop duplicate rows based on all columns. Consider a subset if specific columns define uniqueness.
# For now, we'll assume a row is duplicate if all its values are the same.
initial_rows = len(merged_df)
merged_df.drop_duplicates(inplace=True)
print(f"\nRemoved {initial_rows - len(merged_df)} duplicate rows.")

# 3. Handle Casing:
# Convert string columns to a consistent casing (e.g., title case for names, lower case for emails).
string_cols = merged_df.select_dtypes(include=['object']).columns
for col in string_cols:
    if col == 'email':
        merged_df[col] = merged_df[col].str.lower()
    elif col in ['name', 'username', 'city', 'country', 'address.city', 'address.street', 'company.name']:
        # Convert to title case, handling potential 'Unknown' values
        merged_df[col] = merged_df[col].apply(lambda x: x.title() if isinstance(x, str) and x != 'Unknown' else x)
    else:
        merged_df[col] = merged_df[col].str.strip()


print("\n--- After Nulls, Duplicates, and Casing Cleaning ---")
print(merged_df.info())
print("\nMissing values after cleaning:")
print(merged_df.isnull().sum()[merged_df.isnull().sum() > 0])
display(merged_df.head())


--- Before Cleaning ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id_api               10 non-null     float64
 1   username             10 non-null     object 
 2   email                17 non-null     object 
 3   phone                10 non-null     object 
 4   website              10 non-null     object 
 5   address.street       10 non-null     object 
 6   address.suite        10 non-null     object 
 7   address.city         10 non-null     object 
 8   address.zipcode      10 non-null     object 
 9   address.geo.lat      10 non-null     object 
 10  address.geo.lng      10 non-null     object 
 11  company.name         10 non-null     object 
 12  company.catchPhrase  10 non-null     object 
 13  company.bs           10 non-null     object 
 14  id_csv               7 non-null      float64
 15  age              

,id_api,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs,id_csv,age,city,country,salary,name
0,9.0,Delphine,chaim_mcdermott@dana.io,(775)976-6794 x41206,conrad.com,Dayna Park,Suite 449,Bartholomebury,76495-3109,24.6463,-168.8889,Yost And Sons,Switchable contextually-based project,aggregate real-time technologies,NaN,32.5,Unknown,Unknown,72500.0,Glenna Reichert
1,4.0,Karianne,julianne.oconner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services,NaN,32.5,Unknown,Unknown,72500.0,Patricia Lebsack
2,6.0,Leopoldo_Corkery,karley_dach@jasper.info,1-477-935-8478 x6430,ola.org,Norberto Crossing,Apt. 950,South Christy,23505-1337,-71.4197,71.7478,Considine-Lockman,Synchronised bottom-line interface,e-enable innovative applications,NaN,32.5,Unknown,Unknown,72500.0,Mrs. Dennis Schulist
3,5.0,Kamren,lucio_hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler Llc,User-centric fault-tolerant solution,revolutionize end-to-end systems,NaN,32.5,Unknown,Unknown,72500.0,Chelsey Dietrich
4,3.0,Samantha,nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,Mckenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications,NaN,32.5,Unknown,Unknown,72500.0,Clementine Bauch


In [73]:
# 4. Handle Data Types:
# Convert columns to appropriate types. 'id_api', 'id_csv', 'age', 'salary' should be numeric.

# Convert 'id_api' to integer, coercing errors to NaN and then filling. Since 'id_api' is a primary key from the API, we can assume its not null after fillna for numerical columns.
merged_df['id_api'] = pd.to_numeric(merged_df['id_api'], errors='coerce').astype('Int64')

# 'id_csv' might have NaN if a user was only in API data, so keep it as nullable integer (Int64).
merged_df['id_csv'] = pd.to_numeric(merged_df['id_csv'], errors='coerce').astype('Int64')

# 'age' and 'salary' were already handled for nulls with median, now ensure they are integer type (or float if decimals are needed).
# Using 'Int64' to allow for NaNs if they were not filled properly or if they are legitimately missing for some records.
# Explicitly round before converting to Int64 to handle potential float values from median.
merged_df['age'] = merged_df['age'].round().astype('Int64')
merged_df['salary'] = merged_df['salary'].round().astype('Int64')

# 5. Handle Whitespace:
# Trim whitespace from all string columns.
for col in merged_df.select_dtypes(include=['object']).columns:
    merged_df[col] = merged_df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)


print("\n--- After Data Type and Whitespace Cleaning ---")
print(merged_df.info())
display(merged_df.head())


--- After Data Type and Whitespace Cleaning ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id_api               10 non-null     Int64 
 1   username             17 non-null     object
 2   email                17 non-null     object
 3   phone                17 non-null     object
 4   website              17 non-null     object
 5   address.street       17 non-null     object
 6   address.suite        17 non-null     object
 7   address.city         17 non-null     object
 8   address.zipcode      17 non-null     object
 9   address.geo.lat      10 non-null     object
 10  address.geo.lng      10 non-null     object
 11  company.name         17 non-null     object
 12  company.catchPhrase  17 non-null     object
 13  company.bs           17 non-null     object
 14  id_csv               7 non-null      Int64 
 15  age       

,id_api,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs,id_csv,age,city,country,salary,name
0,9,Delphine,chaim_mcdermott@dana.io,(775)976-6794 x41206,conrad.com,Dayna Park,Suite 449,Bartholomebury,76495-3109,24.6463,-168.8889,Yost And Sons,Switchable contextually-based project,aggregate real-time technologies,<NA>,32,Unknown,Unknown,72500,Glenna Reichert
1,4,Karianne,julianne.oconner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services,<NA>,32,Unknown,Unknown,72500,Patricia Lebsack
2,6,Leopoldo_Corkery,karley_dach@jasper.info,1-477-935-8478 x6430,ola.org,Norberto Crossing,Apt. 950,South Christy,23505-1337,-71.4197,71.7478,Considine-Lockman,Synchronised bottom-line interface,e-enable innovative applications,<NA>,32,Unknown,Unknown,72500,Mrs. Dennis Schulist
3,5,Kamren,lucio_hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler Llc,User-centric fault-tolerant solution,revolutionize end-to-end systems,<NA>,32,Unknown,Unknown,72500,Chelsey Dietrich
4,3,Samantha,nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,Mckenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications,<NA>,32,Unknown,Unknown,72500,Clementine Bauch


In [74]:
# 6. Handle Outliers:
# For numerical columns ('age', 'salary'), use IQR method to cap outliers.

def cap_outliers_iqr(df, column):
    # Create a temporary series that is strictly float64 and without NaNs for calculation and clipping
    # This step is crucial to ensure all elements are numerical and avoid type inference issues
    temp_numeric_series = pd.to_numeric(df[column], errors='coerce')

    # Calculate median using the temporary series (might still contain NaNs if coercion happened)
    median_val = temp_numeric_series.median()

    # Fill NaNs with median *before* quantile calculation and clipping
    temp_numeric_series_filled = temp_numeric_series.fillna(median_val)

    Q1 = temp_numeric_series_filled.quantile(0.25)
    Q3 = temp_numeric_series_filled.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Apply clipping to the temporary, fully numeric and filled series
    # The result will be float64
    clipped_series_float = temp_numeric_series_filled.clip(lower=lower_bound, upper=upper_bound)

    # Assign the clipped (float) values back to the original DataFrame column.
    # If the original column was intended to be Int64, convert back.
    # Use .round() for clean conversion from float to integer.
    df[column] = clipped_series_float.round().astype('Int64')
    return df

numerical_cols_for_outliers = ['age', 'salary']

print("\n--- Before Outlier Handling (first 5 rows with age and salary) ---")
display(merged_df[['age', 'salary']].head())

for col in numerical_cols_for_outliers:
    if col in merged_df.columns:
        merged_df = cap_outliers_iqr(merged_df, col)

print("\n--- After Outlier Handling (first 5 rows with age and salary) ---")
display(merged_df[['age', 'salary']].head())

print("\nAll 6 cleaning techniques applied. Final DataFrame info:")
print(merged_df.info())
print("\nMissing values after all cleaning:")
print(merged_df.isnull().sum()[merged_df.isnull().sum() > 0])
display(merged_df.head())


--- Before Outlier Handling (first 5 rows with age and salary) ---


,age,salary
0,32,72500
1,32,72500
2,32,72500
3,32,72500
4,32,72500



--- After Outlier Handling (first 5 rows with age and salary) ---


,age,salary
0,32,72500
1,32,72500
2,32,72500
3,32,72500
4,32,72500



All 6 cleaning techniques applied. Final DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id_api               10 non-null     Int64 
 1   username             17 non-null     object
 2   email                17 non-null     object
 3   phone                17 non-null     object
 4   website              17 non-null     object
 5   address.street       17 non-null     object
 6   address.suite        17 non-null     object
 7   address.city         17 non-null     object
 8   address.zipcode      17 non-null     object
 9   address.geo.lat      10 non-null     object
 10  address.geo.lng      10 non-null     object
 11  company.name         17 non-null     object
 12  company.catchPhrase  17 non-null     object
 13  company.bs           17 non-null     object
 14  id_csv               7 non-null      Int64 
 15  a

,id_api,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs,id_csv,age,city,country,salary,name
0,9,Delphine,chaim_mcdermott@dana.io,(775)976-6794 x41206,conrad.com,Dayna Park,Suite 449,Bartholomebury,76495-3109,24.6463,-168.8889,Yost And Sons,Switchable contextually-based project,aggregate real-time technologies,<NA>,32,Unknown,Unknown,72500,Glenna Reichert
1,4,Karianne,julianne.oconner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services,<NA>,32,Unknown,Unknown,72500,Patricia Lebsack
2,6,Leopoldo_Corkery,karley_dach@jasper.info,1-477-935-8478 x6430,ola.org,Norberto Crossing,Apt. 950,South Christy,23505-1337,-71.4197,71.7478,Considine-Lockman,Synchronised bottom-line interface,e-enable innovative applications,<NA>,32,Unknown,Unknown,72500,Mrs. Dennis Schulist
3,5,Kamren,lucio_hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler Llc,User-centric fault-tolerant solution,revolutionize end-to-end systems,<NA>,32,Unknown,Unknown,72500,Chelsey Dietrich
4,3,Samantha,nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,Mckenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications,<NA>,32,Unknown,Unknown,72500,Clementine Bauch
